# Cloud Enhancement Event Detection with REST2
# 基于 REST2 的云增强事件检测

REST2 clear-sky modeling, QC (closure, diffuse ratio, tracker-off), and Wang cloud enhancement event (CEE) detection.

REST2 晴空建模、QC（闭合、散射比、跟踪器失准）与 Wang 云增强事件检测。

Assumes the `bsrn` package is installed via pip.
假定 `bsrn` 包已通过 pip 安装。

## 1. Setup / 配置

In [1]:
import os
import sys
import pandas as pd
import bsrn

# Config: input file, output path. REST2 fetches MERRA-2 from Hugging Face (dazhiyang/bsrn-merra2) into RAM.
# 配置：输入文件、输出路径。REST2 从 Hugging Face (dazhiyang/bsrn-merra2) 获取 MERRA-2 到内存。
INPUT_FILE = "../../data/QIQ/qiq0124.dat.gz"
OUTPUT_FILE = "qiq_cee_results.csv"

## 2. Read QIQ data / 读取 QIQ 数据

In [2]:
df = bsrn.io.readers.read_station_to_archive(INPUT_FILE, logical_records="lr0100")
if df is None:
    raise SystemExit(f"Failed to read {INPUT_FILE}")
df.head()

,ghi,bni,dhi,lwd,temp,rh,pressure
time,,,,,,,
2024-01-01 00:00:00+00:00,33.0,72.0,28.0,189.0,-12.1,74.0,995.0
2024-01-01 00:01:00+00:00,36.0,83.0,29.0,189.0,-12.2,73.0,995.0
2024-01-01 00:02:00+00:00,39.0,102.0,31.0,189.0,-12.2,73.0,995.0
2024-01-01 00:03:00+00:00,40.0,100.0,32.0,189.0,-12.3,73.0,995.0
2024-01-01 00:04:00+00:00,40.0,83.0,34.0,189.0,-12.3,74.0,995.0


## 3. REST2 clear-sky modeling / REST2 晴空建模

MERRA-2 parquet is fetched from Hugging Face into RAM (no disk cache). You will see `Fetching MERRA-2 from Hugging Face: qiq/qiq0124_merra2.parquet` when this cell runs.
MERRA-2 parquet 从 Hugging Face 获取到内存（无磁盘缓存）。运行此单元格时会看到 `Fetching MERRA-2 from Hugging Face: qiq/qiq0124_merra2.parquet`。

In [3]:
df = bsrn.modeling.clear_sky.add_clearsky_columns(
    df, station_code="QIQ", model="rest2"
)
df[["ghi", "ghi_clear", "bni", "bni_clear", "dhi", "dhi_clear"]].head()

Fetching MERRA-2 from Hugging Face: qiq/qiq0124_merra2.parquet


,ghi,ghi_clear,bni,bni_clear,dhi,dhi_clear
time,,,,,,
2024-01-01 00:00:00+00:00,33.0,29.556179,72.0,159.759526,28.0,21.102296
2024-01-01 00:01:00+00:00,36.0,31.165183,83.0,166.959123,29.0,21.956312
2024-01-01 00:02:00+00:00,39.0,32.792222,102.0,174.103865,31.0,22.800400
2024-01-01 00:03:00+00:00,40.0,34.436816,100.0,181.189548,32.0,23.634844
2024-01-01 00:04:00+00:00,40.0,36.098472,83.0,188.212621,34.0,24.459916


## 4. QC: closure, diffuse ratio, tracker-off / QC：闭合、散射比、跟踪器失准

**Why PPL and ERL are not used:** Level 1 (PPL) and Level 2 (ERL) QC tests apply physicallt possible and extremely rare bounds that may filter out valid CEE data. CEEs are characterized by GHI exceeding clear-sky or extraterrestrial irradiance—values that PPL/ERL can flag as outliers. This pipeline uses closure, diffuse ratio, and tracker-off checks instead, which are less likely to remove CEE events.

**为何不用 PPL 和 ERL：** 第 1 级（PPL）和第 2 级（ERL）QC 测试会施加物理和气候学界限，可能过滤掉有效的 CEE 数据。CEE 的特征是 GHI 超过晴空或地外辐照度，这些值可能被 PPL/ERL 标为异常。本流水线改用闭合、散射比和跟踪器失准检查，不易剔除 CEE 事件。

In [4]:
df = bsrn.qc.wrapper.run_qc(
    df,
    station_code="QIQ",
    tests=("closure", "diff_ratio", "tracker"),
)
df[[c for c in df.columns if c.startswith("flag")]].head()

,flag3lowSZA,flag3highSZA,flagKKt,flagKlowSZA,flagKhighSZA,flagTracker
time,,,,,,
2024-01-01 00:00:00+00:00,0,0,0,0,0,0
2024-01-01 00:01:00+00:00,0,0,0,0,0,0
2024-01-01 00:02:00+00:00,0,0,0,0,0,0
2024-01-01 00:03:00+00:00,0,0,0,0,0,0
2024-01-01 00:04:00+00:00,0,0,0,0,0,0


## 5. Row-wise: mask ghi, dhi, bni to NaN where any QC fails
## 5. 按行：若任一 QC 失败则 ghi、dhi、bni 置为 NaN

In [5]:
qc_flags = [
    "flag3lowSZA", "flag3highSZA",
    "flagKKt", "flagKlowSZA", "flagKhighSZA",
    "flagTracker",
]
fail_mask = pd.Series(False, index=df.index)
for col in qc_flags:
    if col in df.columns:
        fail_mask = fail_mask | (df[col] == 1)

for col in ("ghi", "dhi", "bni"):
    if col in df.columns:
        df.loc[fail_mask, col] = float("nan")

print(f"Rows masked: {fail_mask.sum()}")

Rows masked: 347


## 6. Drop unused columns; keep measured + clear-sky / 6. 删除无用列，保留实测与晴空列

In [6]:
keep = ["ghi", "bni", "dhi", "ghi_clear", "bni_clear", "dhi_clear"]
df = df[[c for c in keep if c in df.columns]].copy()
df.head()

,ghi,bni,dhi,ghi_clear,bni_clear,dhi_clear
time,,,,,,
2024-01-01 00:00:00+00:00,33.0,72.0,28.0,29.556179,159.759526,21.102296
2024-01-01 00:01:00+00:00,36.0,83.0,29.0,31.165183,166.959123,21.956312
2024-01-01 00:02:00+00:00,39.0,102.0,31.0,32.792222,174.103865,22.800400
2024-01-01 00:03:00+00:00,40.0,100.0,32.0,34.436816,181.189548,23.634844
2024-01-01 00:04:00+00:00,40.0,83.0,34.0,36.098472,188.212621,24.459916


## 7. Wang CEE detection / Wang 云增强事件检测

In [8]:
out_cee = bsrn.utils.cee_detection.detect_cee(
    "wang",
    ghi=df["ghi"],
    ghi_clear=df["ghi_clear"],
    bni=df["bni"],
    bni_clear=df["bni_clear"],
    dhi=df["dhi"],
    dhi_clear=df["dhi_clear"],
    times=df.index,
)
df["cee_flag"] = out_cee["cee_flag"].values

# Count CEEs: 1-min points flagged, and distinct events (contiguous runs)
cee_flag = out_cee["cee_flag"]
n_cee_points = (cee_flag == 1).sum()
# Count event starts: transition from non-CEE to CEE
starts = (cee_flag == 1) & (cee_flag.shift(1, fill_value=0) != 1)
n_cee_events = int(starts.sum())
print(f"CEE count: {n_cee_points} 1-min points in {n_cee_events} distinct events")

out_cee.head()

CEE count: 88 1-min points in 22 distinct events


,is_enhancement,cee_flag,method
time,,,
2024-01-01 00:00:00+00:00,False,0.0,wang
2024-01-01 00:01:00+00:00,False,0.0,wang
2024-01-01 00:02:00+00:00,False,0.0,wang
2024-01-01 00:03:00+00:00,False,0.0,wang
2024-01-01 00:04:00+00:00,False,0.0,wang


## 8. Save to CSV / 保存为 CSV

In [7]:
df.to_csv(OUTPUT_FILE)
print(f"Saved {len(df)} rows to {OUTPUT_FILE}")

Saved 44640 rows to qiq_cee_results.csv
